# 03b — Prophet: Store-Level Demand Baselines & Project Conclusion

**Goal:** Two objectives in one notebook.

**Part 1 — Store-Level Demand Baselines (Section 5–6):** Fit Prophet
independently on the top 3 stores by revenue — CA_3, CA_1, TX_2.
Store-level modeling demonstrates that the univariate statistical approach
generalises across the demand hierarchy: platform aggregate → store →
individual product-store. It also reveals where the univariate ceiling
binds most tightly — TX_2 shows the clearest case where external demand
drivers (SNAP, price) are required to close the approximation gap.

**Part 2 — Conclusion (Section 10–13):** A structured synthesis of all
results across notebooks 2 and 3. Business question restated in terms of
findings, master comparison table, signal ceiling confirmation, limitations,
and the specific gaps the XGBoost layer is designed to close.

> **Demand proxy reminder:** Observed sales are used throughout as a proxy
> for latent demand. True demand is unobservable without inventory and
> stockout records. All model outputs are demand approximations that support
> inventory decisions, not exact demand recovery.

**Inputs:** `monthly_aggregate.csv`, `sell_prices.csv`, `sales_train_validation.csv`,
`calendar.csv` (raw files reconstructed for store-level aggregation)
**Stores:** CA_3 (17.1% revenue share), CA_1 (12.0%), TX_2 (10.9%)
**Split:** 48 months train / 12 months test — consistent with all prior notebooks
**Prophet config:** Per-store CV grid search — each store tuned independently.
Aggregate parameters (cps=0.5, multiplicative) are NOT assumed to transfer
across hierarchy levels.
**Evaluation order:** (1) Grid search on train only → (2) Out-of-sample test forecast

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import sys
sys.path.append(r"C:\Apps\Expense-Time-Series")

from src.helpers import prophet_cv_search

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)

np.random.seed(42)

# Constants — consistent across all notebooks
TRAIN_MONTHS     = 48
TEST_MONTHS      = 15
FORECAST_HORIZON = 12
REP_SERIES       = 'FOODS_3_163_CA_3_validation'
RAW_DIR          = '../data/raw'
PROCESSED_DIR    = '../data/processed'

# Top 3 stores by revenue (confirmed in EDA Section 14)
TARGET_STORES = ['CA_3', 'CA_1', 'TX_2']

def evaluate(actual, predicted, label):
    """Compute RMSE, MAE, MAPE and print a formatted summary."""
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / np.where(actual == 0, 1e-9, actual))) * 100
    print(f'{label}')
    print(f'  RMSE: ${rmse:>12,.2f}')
    print(f'  MAE:  ${mae:>12,.2f}')
    print(f'  MAPE: {mape:>11.2f}%')
    print()
    return {'label': label, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

def to_prophet(df, date_col='month_dt', target_col='total_revenue'):
    """Rename columns to Prophet's required ds / y format."""
    return df[[date_col, target_col]].rename(
        columns={date_col: 'ds', target_col: 'y'}
    )

print('All imports successful.')

## 2. Build Store-Level Monthly Revenue Series

We reconstruct monthly revenue per store from raw files using the same
memory-efficient aggregation strategy established in the EDA: aggregate
units to the item-store-week level, join prices there rather than on the
full 58M row frame, then roll up to monthly. We filter to the three target
stores before joining prices, keeping the working frame manageable.


In [ ]:
# Load raw files
print('Loading raw files...')
sales_wide = pd.read_csv(f'{RAW_DIR}/sales_train_validation.csv')
calendar   = pd.read_csv(f'{RAW_DIR}/calendar.csv')
prices     = pd.read_csv(f'{RAW_DIR}/sell_prices.csv')

print(f'  sales_train_validation: {sales_wide.shape}')
print(f'  calendar:               {calendar.shape}')
print(f'  sell_prices:            {prices.shape}')
print()

# Filter sales to target stores before melt — dramatically reduces frame size
sales_filtered = sales_wide[sales_wide['store_id'].isin(TARGET_STORES)].copy()
print(f'Rows after filtering to {TARGET_STORES}: {len(sales_filtered):,}')

# Melt wide → long
id_cols  = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in sales_filtered.columns if c.startswith('d_')]

long = sales_filtered.melt(
    id_vars=id_cols, value_vars=day_cols,
    var_name='d', value_name='units_sold'
)
long['units_sold'] = long['units_sold'].astype('int16')
print(f'Long format shape (3 stores): {long.shape}')

# Join calendar — keep only what we need
cal_cols = ['d', 'date', 'wm_yr_wk', 'event_name_1', 'snap_CA', 'snap_TX', 'snap_WI']
cal_slim = calendar[calendar['d'].isin(long['d'].unique())][cal_cols].copy()

long = long.merge(cal_slim, on='d', how='left')
long['date']     = pd.to_datetime(long['date'])
long['month_dt'] = long['date'].dt.to_period('M').dt.to_timestamp()

# Aggregate to item-store-month-week (price join key)
weekly_store = (
    long.groupby(['store_id', 'item_id', 'month_dt', 'wm_yr_wk'])['units_sold']
    .sum().reset_index()
)

# Join prices at weekly level — memory-safe
weekly_store = weekly_store.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
weekly_store['sell_price'] = weekly_store['sell_price'].fillna(0).astype('float32')
weekly_store['revenue']    = (weekly_store['units_sold'] * weekly_store['sell_price']).astype('float32')

# Roll up to monthly per store
store_monthly = (
    weekly_store.groupby(['store_id', 'month_dt'])['revenue']
    .sum().reset_index()
    .rename(columns={'revenue': 'total_revenue'})
    .sort_values(['store_id', 'month_dt'])
    .reset_index(drop=True)
)

# Drop incomplete Jan 2011 (3 days only)
store_monthly = store_monthly[store_monthly['month_dt'] >= '2011-02-01'].reset_index(drop=True)

print()
print('Monthly revenue per store (sample):')
print(store_monthly.groupby('store_id').agg(
    months=('month_dt', 'nunique'),
    total_revenue=('total_revenue', 'sum')
).assign(pct=lambda x: (x['total_revenue'] / x['total_revenue'].sum() * 100).round(1))
)

The reconstruction is consistent with EDA findings:

- **CA_3** is the highest observed demand store — roughly 40% of combined
  three-store revenue, consistent with its 17.1% platform share from
  EDA Section 14. Higher revenue here reflects higher demand density —
  more frequent, higher-volume transactions — making it the most tractable
  store for statistical demand modeling.
- **CA_1** is second and **TX_2** third — the demand ranking within this
  three-store slice mirrors the platform-wide ordering.
- **63 months** per store after dropping the incomplete Jan 2011 observation —
  identical observed demand history to the aggregate and representative series.
- Working frame for three stores is a fraction of the 58M row full frame,
  confirming that pre-filtering before the melt is the right memory strategy.
  No demand signal is lost by this filtering — we are selecting stores,
  not sampling observations within stores.

## 3. Train / Test Split — All Three Stores

We apply the same 48/12 split used in notebooks 2 and 3 to each store's
monthly series. The split is time-based and identical across all stores,
so all store-level forecasts are evaluated on the same held-out period
(Feb 2015 → Jan 2016). This makes cross-store error comparisons directly
meaningful.

In [ ]:
store_splits = {}

for store in TARGET_STORES:
    s = store_monthly[store_monthly['store_id'] == store].reset_index(drop=True)
    train = s.iloc[:TRAIN_MONTHS].copy()
    test  = s.iloc[TRAIN_MONTHS:TRAIN_MONTHS + TEST_MONTHS].copy()
    store_splits[store] = {'train': train, 'test': test, 'full': s}
    print(f'{store}:')
    print(f'  Train: {len(train)} months  ({train["month_dt"].min().date()} → {train["month_dt"].max().date()})')
    print(f'  Test:  {len(test)} months   ({test["month_dt"].min().date()} → {test["month_dt"].max().date()})')
    print(f'  Train mean revenue: ${train["total_revenue"].mean():>10,.0f}/month')
    print()

# Plot all three train/test series side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, store in zip(axes, TARGET_STORES):
    train = store_splits[store]['train']
    test  = store_splits[store]['test']

    connector    = train.iloc[[-1]]
    test_connect = pd.concat([connector, test], ignore_index=True)

    ax.plot(train['month_dt'],        train['total_revenue'],        color='steelblue', linewidth=2, label='Train')
    ax.plot(test_connect['month_dt'], test_connect['total_revenue'], color='orange',    linewidth=2, label='Test')
    split_line = test['month_dt'].min() - pd.Timedelta(days=30)
    ax.axvline(split_line, color='red', linestyle='--', linewidth=1.5, label='Split')
    ax.set_title(f'Store {store}', fontsize=12)
    ax.set_xlabel('Month')
    ax.set_ylabel('Revenue ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.legend(fontsize=9)

plt.suptitle('Train / Test Split — Top 3 Stores (48 months train, 12 months test)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

All three stores display the same structural pattern as the platform aggregate:

- **Strong upward trend in observed demand** from 2011 to 2016, consistent
  across all three stores. No structural breaks or sudden drops — each store
  has a clean, continuous observed demand history suitable for statistical
  baseline modeling.
- **Observed demand levels differ significantly across stores** — CA_3 runs at
  roughly double TX_2's monthly revenue. This reflects genuine differences in
  store size, local demographics, and product range — not just noise. Store-level
  models must learn each store's demand baseline independently; a single shared
  model without store encoding would systematically misestimate both stores.
- **Seasonal demand variation is visually subtle** at the store level — the
  upward trend dominates, consistent with the aggregate finding. The Fourier
  seasonality terms in Prophet will recover the annual pattern internally.
- **The split point (Jan/Feb 2015) is clean** — no discontinuity at the
  boundary confirms the train/test split does not cut through an external
  demand event or structural shift. The evaluation is fair.

## 4. Build Holiday Dataframe

We reuse the holiday configuration from notebook 3: event windows informed
by the EDA holiday impact analysis (Section 11). Store-closed events
(Christmas, Thanksgiving) get negative lead windows to capture the
pre-closure shopping surge; pre-gathering events (SuperBowl, LaborDay)
get a positive lag. The full dataframe covering both train and test periods
is passed to Prophet so it can model holiday effects during the forecast
horizon as well as the training window.

In [ ]:
calendar['date'] = pd.to_datetime(calendar['date'])

window_map = {
    'Christmas':      {'lower_window': -3, 'upper_window':  0},
    'Thanksgiving':   {'lower_window': -3, 'upper_window':  0},
    'NewYear':        {'lower_window': -1, 'upper_window':  0},
    'SuperBowl':      {'lower_window':  0, 'upper_window':  1},
    'LaborDay':       {'lower_window':  0, 'upper_window':  1},
    'Easter':         {'lower_window': -1, 'upper_window':  1},
    'OrthodoxEaster': {'lower_window': -1, 'upper_window':  1},
}

# Full holiday dataframe — covers train + test period
cal_events = calendar[
    (calendar['date'] >= '2011-02-01') &
    calendar['event_name_1'].notna()
][['date', 'event_name_1']].drop_duplicates()

holiday_df = (
    cal_events
    .rename(columns={'date': 'ds', 'event_name_1': 'holiday'})
    .reset_index(drop=True)
)
holiday_df['lower_window'] = holiday_df['holiday'].map(
    lambda x: window_map.get(x, {}).get('lower_window', 0)
)
holiday_df['upper_window'] = holiday_df['holiday'].map(
    lambda x: window_map.get(x, {}).get('upper_window', 0)
)

print(f'Holiday dataframe: {len(holiday_df)} rows, {holiday_df["holiday"].nunique()} unique events')
print(f'Date range: {holiday_df["ds"].min().date()} → {holiday_df["ds"].max().date()}')
print()
print(holiday_df.groupby('holiday')[['lower_window', 'upper_window']].first().to_string())

## 5. Fit Prophet — Each Store

### Configuration Rationale

We run a **per-store CV grid search** rather than carrying forward the
aggregate parameters. The aggregate finding (cps=0.5, multiplicative) was
calibrated on a very different signal — 30,490 series summed together.
Individual store demand series have different noise profiles, different
trend slopes, and potentially different seasonal characteristics. Assuming
parameters transfer across hierarchy levels would be an untested assumption
that could degrade demand approximation quality for specific stores.

The one deliberate structural choice is to fit **three independent models**
rather than a single shared model. Each store has a distinct observed demand
baseline, trend slope, and local demand drivers. Forcing them into a single
Prophet model would require store dummies and interaction terms that add
complexity without adding signal at this stage — Prophet's univariate design
is the right tool for store-level statistical baselines. Cross-store demand
relationships and shared feature encoding are handled by the XGBoost layer
in notebook 4.

In [ ]:
# ── BLOCK A: Per-store grid search on training data only ─────────────────
# We do NOT carry forward cps=0.5/multiplicative from the aggregate.
# Store-level series have different noise profiles — each store gets its own CV.
# Same logic as notebook 3: grid search first, test set untouched until Block C.

param_grid = [
    {'changepoint_prior_scale': cps, 'seasonality_mode': mode}
    for cps  in [0.01, 0.05, 0.1, 0.3, 0.5]
    for mode in ['additive', 'multiplicative']
]

store_best_params = {}

for store in TARGET_STORES:
    print(f'{"="*55}')
    print(f'Grid search — Store {store}')
    print(f'{"="*55}')

    train_p = to_prophet(store_splits[store]['train'])

    cv_results = prophet_cv_search(
        train_p,
        param_grid,
        initial = '730 days',   # 24 months min training window
        period  =  '90 days',   # advance cutoff 3 months at a time
        horizon = '365 days'    # evaluate on 12-month horizon
    )

    best = cv_results.iloc[0]
    store_best_params[store] = {
        'changepoint_prior_scale': best['changepoint_prior_scale'],
        'seasonality_mode':        best['seasonality_mode'],
    }

    print(f'\nTop 3 configurations:')
    print(cv_results.head(3).to_string(index=False))
    print(f'\nSelected: cps={best["changepoint_prior_scale"]}  '
          f'mode={best["seasonality_mode"]}  '
          f'CV RMSE=${best["rmse"]:,.0f}  CV MAPE={best["mape"]:.2f}%')
    print()

print('Grid search complete. Test set still untouched.')
print()
print('Selected parameters per store:')
for store, params in store_best_params.items():
    print(f'  {store}: cps={params["changepoint_prior_scale"]}  mode={params["seasonality_mode"]}')

### Grid Search Results — Key Findings

All three stores selected `cps=0.01` or `cps=0.05` — the stiff end of the
grid. This is the opposite of the aggregate result (cps=0.5) and is the
correct outcome: individual store demand trajectories have more noise and
less momentum than the 30,490-series aggregate. A flexible trend at the
store level would overfit to month-to-month demand variation rather than
recovering the underlying trend.

**CA_3:** `cps=0.01, multiplicative` — CV MAPE 4.3%. The smoothest observed
demand trajectory of the three stores. A stiff trend correctly ignores
short-term demand noise; multiplicative seasonality correctly scales the
seasonal swing with the growing demand baseline.

**CA_1:** `cps=0.01, multiplicative` — CV MAPE 4.9%. Same configuration
as CA_3, consistent with similar structural demand characteristics.
Critically: CV MAPE of 4.9% is measured on rolling windows *within*
the training period (2011–2014). This tells us the model approximates
the training demand trajectory well. If test MAPE comes back substantially
higher, that reflects a demand regime shift in 2015 — not a parameter
problem that could have been fixed with any configuration choice available
from training data alone.

**TX_2:** `cps=0.05, additive` — CV MAPE 11.4%. Notably higher CV error
than the California stores even within the training window. TX_2's observed
demand is more volatile month-to-month — likely driven by SNAP distribution
timing that Prophet cannot encode without an explicit `snap_TX` feature.
This is the clearest store-level case where a univariate statistical baseline
is structurally insufficient and the XGBoost feature layer will matter most.

**Important caveat on CV metrics:** CV RMSE and MAPE here are computed on
rolling 12-month horizons within the 48-month training window. The absolute
values are not comparable to held-out test metrics — they are evaluated on
different time periods at different demand levels. Use them only to rank
configurations within each store's grid search, not as performance guarantees.

## 6. Fit Final Models and Forecast

We fit each store's final Prophet model on the full 48-month training set
using the CV-selected hyperparameters, then forecast 12 months forward.
The test set is touched here for the first time.

In [ ]:
store_forecasts    = {}
store_test_metrics = {}

for store in TARGET_STORES:
    print(f'{"="*55}')
    print(f'Store {store} — Final fit and forecast')
    print(f'{"="*55}')

    train   = store_splits[store]['train']
    test    = store_splits[store]['test']
    train_p = to_prophet(train)
    test_p  = to_prophet(test)
    params  = store_best_params[store]

    # Fit on full training data
    model = Prophet(
        changepoint_prior_scale = params['changepoint_prior_scale'],
        seasonality_prior_scale = params.get('seasonality_prior_scale', 1.0),
        holidays_prior_scale    = 10.0,
        seasonality_mode        = params['seasonality_mode'],
        changepoint_range       = params.get('changepoint_range', 0.8),
        yearly_seasonality      = True,
        weekly_seasonality      = False,
        daily_seasonality       = False,
        interval_width          = 0.95,
        uncertainty_samples     = 1000,
        holidays                = holiday_df
    )
    model.add_seasonality(
        name='quarterly', period=91.25, fourier_order=3,
        mode=params['seasonality_mode']
    )
    model.fit(train_p)

    # Forecast — test period only
    future   = model.make_future_dataframe(
        periods=FORECAST_HORIZON, freq='MS', include_history=False
    )
    forecast = model.predict(future)
    fc_test  = (
        forecast[forecast['ds'].isin(test_p['ds'].values)]
        .iloc[:FORECAST_HORIZON]
        .reset_index(drop=True)
    )

    actual    = test_p['y'].iloc[:FORECAST_HORIZON].values
    predicted = fc_test['yhat'].values

    result = evaluate(actual, predicted, f'Store {store} — Prophet')
    store_test_metrics[store] = result
    store_forecasts[store] = {
        'model': model, 'forecast': forecast,
        'fc_test': fc_test, 'train_p': train_p, 'test_p': test_p
    }

    # Month-by-month table
    print(f'{"Month":<15} {"Forecast":>12} {"Actual":>12} {"Error":>12} {"Error %":>10}')
    print('-' * 65)
    for i in range(FORECAST_HORIZON):
        month   = test_p['ds'].iloc[i]
        fc_val  = fc_test['yhat'].iloc[i]
        act_val = test_p['y'].iloc[i]
        err     = act_val - fc_val
        err_pct = abs(err) / act_val * 100 if act_val != 0 else 0
        flag    = '  ←' if err_pct > 15 else ''
        print(f'{str(month.date()):<15} ${fc_val:>11,.0f} ${act_val:>11,.0f} '
              f'${err:>11,.0f} {err_pct:>9.1f}%{flag}')

    # Forecast plot
    connector    = pd.DataFrame({'ds': [train_p['ds'].iloc[-1]],
                                  'y':  [train_p['y'].iloc[-1]]})
    test_connect = pd.concat([connector, test_p.iloc[:FORECAST_HORIZON]],
                              ignore_index=True)
    fc_connect   = pd.concat([
        pd.DataFrame({'ds': [train_p['ds'].iloc[-1]],
                      'yhat':       [train_p['y'].iloc[-1]],
                      'yhat_lower': [train_p['y'].iloc[-1]],
                      'yhat_upper': [train_p['y'].iloc[-1]]}),
        fc_test[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    ], ignore_index=True)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(train_p['ds'],      train_p['y'],       color='steelblue', linewidth=2, label='Train')
    ax.plot(test_connect['ds'], test_connect['y'],  color='orange',    linewidth=2, label='Actual (test)')
    ax.plot(fc_connect['ds'],   fc_connect['yhat'], color='green',     linewidth=2,
            linestyle='--', label=f'Forecast (cps={params["changepoint_prior_scale"]}, {params["seasonality_mode"]})')
    ax.fill_between(fc_connect['ds'],
                    fc_connect['yhat_lower'], fc_connect['yhat_upper'],
                    color='green', alpha=0.15, label='95% CI')
    ax.set_title(f'Store {store} — Prophet Forecast  '
                 f'MAPE: {result["MAPE"]:.1f}%  RMSE: ${result["RMSE"]:,.0f}', fontsize=12)
    ax.set_xlabel('Month')
    ax.set_ylabel('Revenue ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
    print()

### Store-level demand baseline results — interpretation

**CA_1: 2.66% MAPE — the best demand approximation in the project.**
Every month lands within 5.1% of observed demand, with January 2016
at just 0.2% error. CA_1 has the smoothest, most consistent observed
demand trajectory of the three stores — exactly the conditions where
a stiff univariate trend with multiplicative seasonality approximates
demand most reliably. When the demand signal is clean and consistent,
statistical baselines perform at their best.

**CA_3: 6.23% MAPE — good approximation with a systematic directional bias.**
The model overshoots observed demand in almost every test month.
CA_3's actual demand growth decelerated slightly in the test period
relative to the training trend — Prophet, trained on the faster rate,
projects higher than actuals. November 2015 is the largest miss at
11.8%, likely reflecting softer-than-expected pre-holiday observed
demand. The consistent direction of the bias (overshoot, not random
error) is informative: it suggests a demand regime change in the test
period that was not foreseeable from training history alone. For a
decision-support system, a known directional bias is more useful than
random error — planners can apply a systematic adjustment.

**TX_2: 15.31% MAPE — the clearest store-level univariate ceiling.**
The model undershoots observed demand every single month, with the
gap widening progressively from 12.9% in February 2015 to 21.7% by
January 2016. This compounding pattern is the signature of a demand
growth rate underestimate: Prophet learned a trajectory from training
data that is slightly slower than what TX_2's actual demand delivered
in the test period. The gap accumulates month by month with no
self-correction mechanism available to a univariate model.

This is a structural limitation of univariate demand approximation,
not a tuning failure. Prophet reads only past observed revenue — it
cannot observe the external demand drivers that are pushing TX_2's
actual demand above the projected baseline. The EDA identified the
most likely driver: TX_2's SNAP uplift is the strongest in the
dataset (+17.2% observed FOODS demand on SNAP days). Prophet has
no access to the SNAP calendar as an explicit input. Price changes
and local promotions are equally invisible.

These are precisely the features the XGBoost layer ingests in
notebook 4. TX_2 is the store where the transition from statistical
baselines to machine learning will matter most — the approximation
gap is large, consistent, and directly attributable to identifiable
external demand drivers that are available as features.

**These limitations motivate the use of machine learning models
(XGBoost) in notebook 4, which can incorporate the SNAP calendar,
pricing signals, and lag-based features to close the demand
approximation gaps that univariate baselines cannot address.**

---

# Part 2 — Project Conclusion

---

## 10. Business question restated in terms of demand approximation results

**The original question:**
> *Given a product's observed sales history, price history, seasonality,
> and upcoming holidays — how well can we approximate demand at different
> levels of the hierarchy? And where does the univariate signal ceiling
> bind, requiring external demand drivers to close the gap?*

We answered this across three hierarchy levels using SARIMA, Prophet,
and naive baselines on a strict held-out 12-month test period.
All results measure approximation of observed demand — which is itself
a proxy for true latent demand. Without stockout and inventory data,
we cannot recover true demand; we can only approximate the demand
signal that observed sales carry.

**Platform aggregate:** Prophet achieves **5.02% MAPE** — for every
dollar of observed demand the approximation is off by 5 cents. Reliable
enough to support budget planning, category-level inventory purchasing,
and capacity decisions. At this level of aggregation the demand signal
is strong and the univariate ceiling is high.

**Store level:** CA_1 hits **2.66% MAPE** — the best demand approximation
in the project. CA_3 lands at **6.23%** with a consistent overshoot bias.
TX_2 reaches **15.31%** — a ceiling driven by unobserved SNAP and price
signals that no univariate model can access. Store-level results are
operationally useful for CA_1 and CA_3; TX_2 requires external features
before its demand approximation is reliable enough for inventory decisions.

**Individual product-store:** SARIMA and Prophet tie at **~22% MAPE**
— a 51% improvement over the naive baseline. The remaining error is
concentrated in three specific months where external demand drivers
(price drops, SNAP timing) dominated. These are not random residuals —
they are the signal the ML layer is designed to capture.

## 11. Master Model Comparison Table

All models evaluated on the same held-out 12-month test period
(Feb 2015 → Jan 2016). All metrics measure approximation of observed
demand. Bold = best demand approximation per series across statistical
baselines. XGBoost targets added as forward-looking benchmarks.

---

### Platform aggregate revenue

| Model | RMSE | MAE | MAPE |
|---|---|---|---|
| Naive | $380,051 | $332,241 | 8.96% |
| SMA(3) | $443,310 | $396,285 | 10.71% |
| SARIMA(2,0,1)(0,1,1)[12] | $277,220 | $252,147 | 6.91% |
| **Prophet — cps=0.5, multiplicative** | **$209,726** | **$178,567** | **5.02%** |

---

### Individual product-store — FOODS_3_163 @ CA_3

| Model | RMSE | MAE | MAPE |
|---|---|---|---|
| Naive | $98.48 | $91.31 | 55.29% |
| SMA(3) | $85.66 | $77.31 | 45.80% |
| **SARIMA(0,0,1)(0,1,1)[12]** | **$54.41** | **$37.39** | **22.22%** |
| Prophet — cps=0.01, additive | $52.80 | $39.68 | 24.25% |

*SARIMA wins on MAPE, Prophet wins on RMSE — effectively a statistical tie.*

---

### Store level — top 3 stores by revenue

| Store | Revenue share | Model | RMSE | MAE | MAPE |
|---|---|---|---|---|---|
| **CA_1** | 12.0% | **Prophet — cps=0.01, multiplicative** | **$13,104** | **$11,313** | **2.66%** |
| CA_3 | 17.1% | Prophet — cps=0.01, multiplicative | $39,078 | $36,472 | 6.23% |
| TX_2 | 10.9% | Prophet — cps=0.05, additive | $56,856 | $55,343 | 15.31% |

*TX_2 represents the clearest store-level univariate ceiling — systematic
demand undershoot driven by unobserved SNAP and price signals that are
not accessible to any univariate model. This is a structural gap, not a
tuning failure. XGBoost target: < 10% MAPE by incorporating `snap_TX`,
`sell_price`, and lag features as explicit demand drivers.*

## 12. Which Model Won and Why

In [ ]:
# Visual summary — MAPE by model and series level
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Aggregate series
agg_models = ['Naive', 'SMA(3)', 'SARIMA', 'Prophet']
agg_mapes  = [8.96, 10.71, 6.91, 5.02]
colors_agg = ['#d62728' if m == 'Prophet' else 'steelblue' for m in agg_models]

bars = axes[0].bar(agg_models, agg_mapes, color=colors_agg, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, agg_mapes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Aggregate Series — MAPE by Model', fontsize=12)
axes[0].set_ylabel('MAPE (%)')
axes[0].set_ylim(0, 13)
axes[0].axhline(agg_mapes[-1], color='#d62728', linestyle='--', linewidth=1, alpha=0.4)

# Representative series
rep_models = ['Naive', 'SMA(3)', 'SARIMA', 'Prophet']
rep_mapes  = [55.29, 45.80, 22.22, 24.25]
colors_rep = ['#d62728' if m == 'SARIMA' else 'steelblue' for m in rep_models]

bars = axes[1].bar(rep_models, rep_mapes, color=colors_rep, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, rep_mapes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('Representative Series — MAPE by Model', fontsize=12)
axes[1].set_ylabel('MAPE (%)')
axes[1].set_ylim(0, 65)
axes[1].axhline(min(rep_mapes), color='#d62728', linestyle='--', linewidth=1, alpha=0.4)

plt.suptitle('Model Performance Summary — Lower MAPE is Better\n(Red bar = winner per series)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Why Prophet produces a better aggregate demand approximation than SARIMA

Single reason: **trend continuation**. Observed aggregate demand roughly
doubled from $2M to $4M per month over 5 years. SARIMA's AR structure
is mean-reverting — as the horizon extends it pulls demand estimates back
toward the historical training mean rather than continuing the observed
trajectory. By month 12, SARIMA's approximation error reaches 10.9%.
Prophet's piecewise linear trend extrapolates the slope forward and
reaches 1.3% error on the same month.

Multiplicative seasonality outperforming additive was confirmed empirically
by CV — as observed aggregate demand doubled, the seasonal amplitude in
observed sales grew proportionally. Additive seasonality assumes a fixed
dollar swing regardless of the demand baseline and systematically
underestimates seasonal peaks in later years. This is a specification
finding, not an assumption.

---

### Why SARIMA and Prophet tie on the individual series — and why it matters

Both reach ~22% MAPE and both produce their largest errors on the exact
same three months — April 2015, May 2015, January 2016 — with errors
exceeding 40%. The approximation failures are identical because the cause
is identical: observed demand in those months was driven by price changes
and SNAP distribution events that neither model can observe from the
time series alone.

This is the **univariate signal ceiling**. It is not a model failure —
it is a precise characterization of the information boundary beyond which
historical time series patterns cannot approximate demand. No amount of
parameter tuning closes it because the missing inputs are not in the
time series. The path forward is different data, not a different model.

## 13. The Signal Ceiling Finding

This is the most important analytical finding of the project and the direct
motivation for the XGBoost layer in notebook 4.

In [ ]:
# Visualise the shared failure months across SARIMA and Prophet
# We reconstruct the month-by-month errors for the representative series
# using hardcoded values from notebook 2 and 3 results

months = pd.date_range('2015-02-01', periods=12, freq='MS')

# SARIMA errors (from notebook 2, Section 5c)
sarima_errors = [16.3, 17.3, 54.9, 47.3, 8.7, 9.0, 6.9, 1.4, 10.7, 10.6, 7.2, 55.6]

# Prophet errors (from notebook 3, Section 4b)
prophet_errors = [20.5, 14.2, 58.5, 49.8, 12.3, 7.1, 11.0, 4.8, 13.2, 9.7, 6.1, 40.0]

fig, ax = plt.subplots(figsize=(14, 5))

x     = np.arange(len(months))
width = 0.35

bars1 = ax.bar(x - width/2, sarima_errors,  width, label='SARIMA',  color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, prophet_errors, width, label='Prophet', color='orange',    alpha=0.85)

# Highlight the three shared failure months
spike_months = [2, 3, 11]  # April 2015, May 2015, January 2016
for i in spike_months:
    ax.axvspan(i - 0.5, i + 0.5, alpha=0.12, color='red')

ax.axhline(20, color='red', linestyle='--', linewidth=1.5, alpha=0.6, label='20% error threshold')
ax.set_xticks(x)
ax.set_xticklabels([m.strftime('%b %Y') for m in months], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Absolute Percentage Error (%)')
ax.set_title(
    'Month-by-Month Error — SARIMA vs Prophet (Representative Series FOODS_3_163_CA_3)\n'
    'Red shading = shared failure months driven by unobserved external features',
    fontsize=12
)
ax.legend(fontsize=10)
ax.set_ylim(0, 70)
plt.tight_layout()
plt.show()

print('Signal ceiling summary:')
print(f'  Months where BOTH models exceed 30% error: Apr 2015, May 2015, Jan 2016')
print(f'  SARIMA errors in those months:  {sarima_errors[2]:.1f}%, {sarima_errors[3]:.1f}%, {sarima_errors[11]:.1f}%')
print(f'  Prophet errors in those months: {prophet_errors[2]:.1f}%, {prophet_errors[3]:.1f}%, {prophet_errors[11]:.1f}%')
print()
print('  Probable causes (from EDA):')
print('    Apr/May 2015 — likely a price drop event (EDA confirmed asymmetric elasticity')
print('                   at FOODS_3 product-store level: 10% price drop → ~77% demand surge)')
print('    Jan 2016     — likely SNAP distribution timing or post-holiday restocking spike')
print()
print('  Why XGBoost fixes this:')
print('    sell_price + price_change_pct features directly encode the price signal')
print('    snap_CA / snap_TX / snap_WI features directly encode the SNAP calendar')
print('    lag_7, lag_28 features encode recent momentum that preceded each spike')

The chart makes the univariate signal ceiling concrete — two different
model families, calibrated independently, producing nearly identical
large errors on the same three months. The ceiling is real, it is
measurable, and it has a clear cause.

**The fix is not a better statistical model. It is different inputs.**

---

### The three gaps the XGBoost layer is designed to close

**Gap 1 — External demand drivers**
SARIMA and Prophet read only past observed revenue. They are blind to
the factors that actually drove demand in the failure months. XGBoost
ingests these as explicit features:

| Feature | Demand signal | EDA evidence |
|---|---|---|
| `sell_price` + `price_change_pct` | Price-driven demand response | r=0.553 for drops at product-store level — ~77% demand increase per 10% price drop |
| `snap_CA / snap_TX / snap_WI` | SNAP-driven demand spikes | +10.3% to +32.5% observed FOODS demand uplift by state |
| `is_event_day` | Event-driven demand shifts | SuperBowl +18.9%, Labor Day +19.6% observed uplift |

These signals were present in the data during the failure months.
They were simply not accessible to univariate models reading only
the revenue series.

**Gap 2 — Recent demand momentum via lag features**
SARIMA's AR terms are fixed estimates from the full training history
and cannot adapt to local demand acceleration. XGBoost lag features
are computed from recent actuals and reflect current demand conditions:

| Feature | What it captures |
|---|---|
| `lag_7` | Observed demand from one week ago |
| `lag_28` | Observed demand from four weeks ago |
| `rolling_mean_7` | Short-term demand momentum |
| `rolling_mean_28` | Medium-term demand baseline |

This is what TX_2 needed: a model that could detect its demand
growth accelerating rather than projecting the slower training-period
rate indefinitely forward.

**Gap 3 — Scale across the full demand hierarchy**
One SARIMA per series is viable only for the 2,469 complete series
(8.1% of the full set). XGBoost trains a single model across all
30,490 product-store combinations simultaneously, using `store_id`
and `dept_id` encodings to learn individual demand baselines within
one shared model. Evaluated via walk-forward cross-validation to
prevent leakage — no future demand data is ever used to train on
any fold.

---

### Demand approximation targets for notebook 4

| Series | Statistical baseline | XGBoost target | Primary driver of expected gain |
|---|---|---|---|
| Aggregate | 5.02% MAPE | Marginal improvement | Prophet already handles trend; limited upside |
| Representative | 22.22% MAPE | < 15% | Price and SNAP features address the exact failure months |
| TX_2 store | 15.31% MAPE | < 10% | `snap_TX` + lag features address the systematic undershoot |

---

### Limitations carried forward into notebook 4

- **Observed sales ≠ true demand.** Revenue is derived (`units × price`)
  with minor gaps where price records are missing. Zero observed sales
  on a given day may reflect a stockout (unmet demand) or genuine demand
  absence — indistinguishable without inventory data. All downstream
  demand approximations carry this limitation.
- Only 8.1% of product-store series have complete 64-month histories.
  XGBoost lag features must not span structural zero gaps — pulling
  signal from periods when a product was unavailable conflates supply
  absence with demand absence.
- Results here are from one representative series at the median observed
  demand level. Approximation quality varies across the full 30,490
  series depending on volume, sparsity, and exposure to SNAP and
  price-driven demand events.
- All statistical models in notebooks 2 and 3 operate on monthly
  aggregated data. XGBoost in notebook 4 targets the daily level —
  where zero-inflation and demand intermittency make the approximation
  problem substantially harder, and where the lag feature engineering
  must be applied most carefully to avoid spanning structural gaps.

**These limitations collectively define the scope and honest claims
of this demand intelligence system: it approximates demand patterns
that support inventory decisions, using observed sales as a proxy
for true latent demand. It does not recover true demand, and it does
not optimize inventory directly.**